In [ ]:
import pandas as pd
from DerivModel import FeedForwardNetwork, train_model
from BasketDataGen import value_basket
from DerivPlots import scatter_plot
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm.notebook import tqdm, trange
import numpy as np
import matplotlib.pyplot as plt
import QuantLib as ql
import time

In [ ]:
model_file = 'modelDMC.pt'

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
input_size = 28
num_hidden_layers = 4
neurons_per_layer = 300
model = FeedForwardNetwork(input_size, num_hidden_layers, neurons_per_layer).to(device)
model = torch.load(model_file, map_location=device, weights_only=False)
model.eval()

In [ ]:
filename_root = 'test_basket_mt'

df1 = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1.columns:
        df1[col] = (
            df1[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )
df1e5 = df1[df1['samples'] == 100000]
df1e5 = df1e5.drop('samples', axis=1)

In [ ]:
df1e5['process_time'].mean()

In [ ]:
df1e4 = df1[df1['samples'] == 10000]
df1e4['process_time'].mean()

In [ ]:
df1e3 = df1[df1['samples'] == 1000]
df1e3['process_time'].mean()

In [ ]:
y = df1e5['option_value']
X = df1e5.drop(['option_value', 'process_time', 'error_estimate'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01) 

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 4096
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, drop_last=True)

In [ ]:
batches = 0
start_time = time.time()
with torch.no_grad():
    for batch_X, batch_y in data_loader:
        model(batch_X)
        batches = batches + 1
end_time = time.time()

In [ ]:
batches

In [ ]:
(end_time - start_time) / (batches * batch_size)

In [ ]:
(end_time - start_time) / (batches)

In [ ]:
batches = 0
start_time = time.time()
with torch.no_grad():
    for batch_X, batch_y in test_data_loader:
        model(batch_X)
        batches = batches + 1
end_time = time.time()

In [ ]:
(end_time - start_time) / (batches)